<a href="https://colab.research.google.com/github/MedhatElhawy/AI-Practices/blob/main/RAGSystem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U langchain langchain-cohere langchain-community langchain-text-splitters langchain-chroma pymupdf chromadb gradio

In [ ]:
import os
import time
import gradio as gr
from langchain_chroma import Chroma
from langchain_cohere import ChatCohere, CohereEmbeddings
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ==========================================
# 1. API Key Setup
# ==========================================
try:
  from google.colab import userdata

  key = userdata.get('coherekey')
except ImportError:
  from dotenv import load_dotenv

  load_dotenv()
  key = os.getenv('coherekey')

embeddings = CohereEmbeddings(model='embed-v4.0', cohere_api_key=key)
llm = ChatCohere(model='command-a-03-2025', temperature=0, cohere_api_key=key)

# ==========================================
# 2. Categories (Lectures + Book + CV)
# ==========================================
pdf_groups = {
    'lecture': [
        '/content/AI314 - Lec 1 - Sp26 - Multi-Agents Intro.pdf',
        '/content/AI314 - Lec 2 - IIntelligent Agent part1 Sp26.pdf',
        '/content/AI314 - Lec 3 - IIntelligent Agent part2 Sp26.pdf',
        (
            '/content/AI314 - Lec 4 - Intelligent Agent and solving problems'
            ' by searching Sp26.pdf'
        ),
        '/content/AI314 - Lec 5 -Search Strategies midterm revision- Sp26.pdf',
        (
            '/content/AI314 - Lec 6 -Uninformed (Blind) Search Strategies-'
            ' Sp26.pdf'
        ),
        (
            '/content/AI314 - Lec 7 - Informed Search Strategies - Heuristics &'
            ' Metaheuristics- Sp26.pdf'
        ),
        (
            '/content/AI314 - Lec 8 Complex local search problems &'
            ' Optimization- Sp26 - part 1.pdf'
        ),
        (
            '/content/AI314 - Lec 9 Complex local search problems &'
            ' Optimization- Sp26 - part 2.pdf'
        ),
        '/content/AI314 - Lec 10 Multi-Agents Environment- Sp26.pdf',
    ],
    'book': [
        (
            '/content/Aurélien-Géron-Hands-On-Machine-Learning-with-Scikit-Learn-Keras-and-Tensorflow_-Concepts-Tools-and-Techniques-to-Build-Intelligent-Systems-O’Reilly-Media-2019.pdf'
        ),
    ],
    'cv': [
        '/content/Medhat_Orabi_CV.pdf',
    ],
}

# 3. Load Documents & Add Category Metadata
all_documents = []
for category, files in pdf_groups.items():
  for file_path in files:
    if os.path.exists(file_path):
      print(f'Loading [{category}]: {os.path.basename(file_path)}')
      loader = PyMuPDFLoader(file_path)
      docs = loader.load()

      for doc in docs:
        doc.metadata['category'] = category
      all_documents.extend(docs)
    else:
      print(f'Warning: File not found -> {file_path}')

# 4. Chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800, chunk_overlap=100
)
chunks = text_splitter.split_documents(all_documents)

# 5. ChromaDB VectorStore
vectorstore = Chroma(
    collection_name='tabbed_pdf_rag', embedding_function=embeddings
)
batch_size = 90
for i in range(0, len(chunks), batch_size):
  batch = chunks[i : i + batch_size]
  vectorstore.add_documents(documents=batch)
  print(
      f'Uploaded batch {i // batch_size + 1} of {((len(chunks) - 1) // batch_size) + 1}'
  )
  time.sleep(12)

# ==========================================
# 6. Retrievers Filtered by Category
# ==========================================
retriever_lectures = vectorstore.as_retriever(
    search_kwargs={'k': 5, 'filter': {'category': 'lecture'}}
)
retriever_book = vectorstore.as_retriever(
    search_kwargs={'k': 5, 'filter': {'category': 'book'}}
)
retriever_cv = vectorstore.as_retriever(
    search_kwargs={'k': 5, 'filter': {'category': 'cv'}}
)
retriever_all = vectorstore.as_retriever(search_kwargs={'k': 5})

# ==========================================
# 7. RAG Chain Generator
# ==========================================
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.
Answer the user's question using ONLY the context below.
If the answer cannot be found in the context, say "I don't know."

Context:
{context}

Question:
{question}
""")


def format_docs(docs):
  return '\n\n'.join(doc.page_content for doc in docs)


def create_rag_handler(target_retriever):
  rag_chain = (
      {
          'context': target_retriever | format_docs,
          'question': RunnablePassthrough(),
      }
      | prompt
      | llm
  )

  def answer_question(question, history):
    retrieved_docs = target_retriever.invoke(question)
    response = rag_chain.invoke(question)
    answer = (
        response.content if hasattr(response, 'content') else str(response)
    )

    sources_info = []
    if answer.strip() != "I don't know." and retrieved_docs:
      sources_seen = set()
      for doc in retrieved_docs:
        src_file = os.path.basename(doc.metadata.get('source', 'Unknown'))
        page_num = doc.metadata.get('page', 0) + 1
        entry = f'📄 File: {src_file} (Page {page_num})'
        if entry not in sources_seen:
          sources_seen.add(entry)
          sources_info.append(entry)

    if sources_info:
      answer += '\n\n---\n**Sources:**\n' + '\n'.join(sources_info)

    return answer

  return answer_question


# ==========================================
# 8. Gradio UI Interfaces
# ==========================================
interface_lectures = gr.ChatInterface(
    fn=create_rag_handler(retriever_lectures),
    title='📘 Search Autonomous Multiagent Lectures',
    description='Ask questions across Lectures 1 to 10.',
)

interface_book = gr.ChatInterface(
    fn=create_rag_handler(retriever_book),
    title='📚 Search Hands-On-Machine-Learning Book',
    description='Ask questions about Hands-On Machine Learning Book.',
)

interface_cv = gr.ChatInterface(
    fn=create_rag_handler(retriever_cv),
    title='📄 Search Resume / CV',
    description="Ask questions about Medhat Orabi's CV.",
)

interface_all = gr.ChatInterface(
    fn=create_rag_handler(retriever_all),
    title='🌐 Unified Search (All PDFs)',
    description='Search across Lectures, ML Book, and CV simultaneously.',
)

demo = gr.TabbedInterface(
    interface_list=[
        interface_lectures,
        interface_book,
        interface_cv,
        interface_all,
    ],
    tab_names=[
        '📘 Lectures Task',
        '📚 ML Book Task',
        '📄 CV Task',
        '🌐 Unified All PDFs',
    ],
    title='🤖 Multi-Task PDF RAG System',
)

if __name__ == '__main__':
  demo.launch(share=True, debug=True)

/tmp/ipykernel_5461/2127292470.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


Loading [lecture]: AI314 - Lec 1 - Sp26 - Multi-Agents Intro.pdf
Loading [lecture]: AI314 - Lec 2 - IIntelligent Agent part1 Sp26.pdf
Loading [lecture]: AI314 - Lec 3 - IIntelligent Agent part2 Sp26.pdf
Loading [lecture]: AI314 - Lec 4 - Intelligent Agent and solving problems by searching Sp26.pdf
Loading [lecture]: AI314 - Lec 5 -Search Strategies midterm revision- Sp26.pdf
Loading [lecture]: AI314 - Lec 6 -Uninformed (Blind) Search Strategies- Sp26.pdf
Loading [lecture]: AI314 - Lec 7 - Informed Search Strategies - Heuristics & Metaheuristics- Sp26.pdf
Loading [lecture]: AI314 - Lec 8 Complex local search problems & Optimization- Sp26 - part 1.pdf
Loading [lecture]: AI314 - Lec 9 Complex local search problems & Optimization- Sp26 - part 2.pdf
Loading [lecture]: AI314 - Lec 10 Multi-Agents Environment- Sp26.pdf
Loading [book]: Aurélien-Géron-Hands-On-Machine-Learning-with-Scikit-Learn-Keras-and-Tensorflow_-Concepts-Tools-and-Techniques-to-Build-Intelligent-Systems-O’Reilly-Media-2019.